In [2]:
import requests
import os

uniprot_id = "P04637"
# Endpoint de la API pública de AlphaFold
api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"

print(f"Consultando la API de AlphaFold para {uniprot_id}...")

# 1. Preguntamos a la API cuál es la URL exacta y actualizada
response = requests.get(api_url)

if response.status_code == 200:
    data = response.json()
    # Extraemos la URL del archivo PDB del primer modelo que nos devuelve
    pdb_url = data[0]['pdbUrl']
    print(f"URL oficial localizada: {pdb_url}")

    # 2. Descargamos el archivo
    print("Descargando estructura 3D...")
    pdb_data = requests.get(pdb_url)

    nombre_archivo = "p53_wildtype.pdb"
    with open(nombre_archivo, "wb") as f:
        f.write(pdb_data.content)

    if os.path.exists(nombre_archivo):
        print(f"¡Éxito! El archivo {nombre_archivo} se ha guardado correctamente.")
else:
    print(f"Error al consultar la API. Código de estado: {response.status_code}")

Consultando la API de AlphaFold para P04637...
URL oficial localizada: https://alphafold.ebi.ac.uk/files/AF-P04637-F1-model_v6.pdb
Descargando estructura 3D...
¡Éxito! El archivo p53_wildtype.pdb se ha guardado correctamente.


In [4]:
import requests

# 1. Obtener la secuencia original (WT) desde UniProt
uniprot_id = "P04637"
fasta_url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
response = requests.get(fasta_url)

# Limpiar el FASTA (quitar el encabezado y unir los saltos de línea)
secuencia_wt = "".join(response.text.split('\n')[1:])

# 2. Aplicar la mutación R175H
# En Python los índices empiezan en 0, así que la posición 175 biológica es el índice 174
pos = 175
aa_original = secuencia_wt[pos - 1]
aa_mutado = "H"

secuencia_mutada = secuencia_wt[:pos-1] + aa_mutado + secuencia_wt[pos:]

print(f"Aminoácido en posición {pos}: {aa_original}")
if aa_original == 'R':
    print("¡Verificación superada! Es una Arginina.\n")
    print("Secuencia mutada (R175H) lista para copiar:")
    print("-" * 50)
    print(secuencia_mutada)
    print("-" * 50)
else:
    print("Error: El aminoácido no coincide. Revisa la secuencia.")

Aminoácido en posición 175: R
¡Verificación superada! Es una Arginina.

Secuencia mutada (R175H) lista para copiar:
--------------------------------------------------
MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGPDEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRHCPHHERCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNSSCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELPPGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPGGSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD
--------------------------------------------------


In [9]:
import Bio.PDB
import py3Dmol

parser = Bio.PDB.PDBParser(QUIET=True)
wt_struct = parser.get_structure("WT", "p53_wildtype.pdb")
mut_struct = parser.get_structure("MUT", "p53_mutante.pdb")

# EXTRAEMOS SOLO EL DOMINIO ESTRUCTURADO (Residuos 94 a 292) PARA ALINEAR
inicio, fin = 94, 292
wt_ca = [atom for atom in wt_struct[0].get_atoms() if atom.get_name() == 'CA' and inicio <= atom.get_parent().get_id()[1] <= fin]
mut_ca = [atom for atom in mut_struct[0].get_atoms() if atom.get_name() == 'CA' and inicio <= atom.get_parent().get_id()[1] <= fin]

# Aplicamos la superposición usando solo esa zona central
super_imposer = Bio.PDB.Superimposer()
super_imposer.set_atoms(wt_ca, mut_ca)
super_imposer.apply(mut_struct.get_models())

print(f"Alineamiento del núcleo estructurado completado. Desviación RMSD: {super_imposer.rms:.3f} Å")

# Guardar y visualizar
io = Bio.PDB.PDBIO()
io.set_structure(mut_struct)
io.save("p53_mutante_alineada.pdb")

view = py3Dmol.view(width=800, height=500)
with open("p53_wildtype.pdb", "r") as f:
    view.addModel(f.read(), "pdb")
view.setStyle({'model': 0}, {'cartoon': {'color': 'blue'}})

with open("p53_mutante_alineada.pdb", "r") as f:
    view.addModel(f.read(), "pdb")
view.setStyle({'model': 1}, {'cartoon': {'color': 'red'}})

view.addStyle({'resi': '175'}, {'stick': {'colorscheme': 'magentaCarbon', 'radius': 0.2}})
view.zoomTo()
view.show()

Alineamiento del núcleo estructurado completado. Desviación RMSD: 1.202 Å


3Dmol.js failed to load for some reason. Please check your browser console for error messages.